# PART 5. 생존율 비교

성별·객실 등급·나이 집단을 비교합니다.

설명 → 코드 → 실행 결과를 확인하세요. 원본은 변경하지 않습니다. Python 패키지는 프로젝트 requirements.txt로 설치하세요. 노트북 출력은 실행 후 생성됩니다.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'STEP3_Local_Deploy' / 'titanic_dashboard').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('프로젝트를 풀어 놓은 폴더 안에서 실행하세요. Colab에서는 프로젝트 폴더 경로를 ROOT로 지정하세요.')
    ROOT = ROOT.parent
APP = ROOT / 'STEP3_Local_Deploy' / 'titanic_dashboard'
sys.path.insert(0, str(APP))
from utils.data import load_data
df = load_data()
print('원본:', df.shape)


원본: (891, 12)


정상 결과: 어린이 113명 중 61명, 성인 601명 중 229명 생존.

In [2]:
from utils.data import summary, age_data
print(summary(age_data(df),"AgeGroup").round(1).to_string(index=False))
print(summary(df,"Sex").round(1).to_string(index=False))
print(summary(df,"Pclass").round(1).to_string(index=False))

AgeGroup  인원  생존자  생존율(%)
 성인(≥18) 601  229    38.1
어린이(<18) 113   61    54.0
   Sex  인원  생존자  생존율(%)
female 314  233    74.2
  male 577  109    18.9
 Pclass  인원  생존자  생존율(%)
      1 216  136    63.0
      2 184   87    47.3
      3 491  119    24.2


### 1. 연령대별 생존율\n나이가 기록된 승객 714명을 연령대별로 묶어 생존율을 비교합니다. 나이가 비어 있는 177명은 계산에서 제외합니다.

In [3]:
import pandas as pd
import plotly.express as px

age_bins = [0, 10, 20, 30, 40, 50, 60, 70, 81]
age_labels = ["0–9", "10–19", "20–29", "30–39", "40–49", "50–59", "60–69", "70–80"]
age_df = df.dropna(subset=["Age"]).copy()
age_df["연령대"] = pd.cut(age_df["Age"], bins=age_bins, labels=age_labels, right=False)
age_summary = (
    age_df.groupby("연령대", observed=False)["Survived"]
    .mean().mul(100).dropna().rename("생존율(%)").reset_index()
)
fig = px.line(age_summary, x="연령대", y="생존율(%)", markers=True,
              title="타이타닉 승객 연령대별 생존율",
              color_discrete_sequence=["#2878B5"])
fig.update_xaxes(title="연령대 (세)")
fig.update_yaxes(title="생존율 (%)", range=[0, 100])
fig.add_annotation(text="출처: train.csv · 나이 미상 177명 제외 (n=714)", xref="paper", yref="paper", x=0, y=-0.2, showarrow=False, xanchor="left", font={"size": 10, "color": "gray"})
fig.update_layout(margin={"b": 70})
fig.show()

### 2. 객실 등급별 생존율\n객실 등급별 생존율은 평균으로 계산하고, 막대그래프의 세로축은 0부터 시작합니다.

In [4]:
class_summary = (
    df.groupby("Pclass")["Survived"].mean().reindex([1, 2, 3])
    .mul(100).rename("생존율(%)").reset_index()
)
class_summary["객실 등급"] = class_summary["Pclass"].map({1: "1등석", 2: "2등석", 3: "3등석"})
fig = px.bar(class_summary, x="객실 등급", y="생존율(%)", text_auto=".1f",
             title="타이타닉 객실 등급별 생존율",
             color="객실 등급", color_discrete_sequence=["#2878B5", "#5AAE8A", "#E39C37"])
fig.update_xaxes(title="객실 등급")
fig.update_yaxes(title="생존율 (%)", range=[0, 100])
fig.add_annotation(text="출처: train.csv (전체 891명)", xref="paper", yref="paper", x=0, y=-0.2, showarrow=False, xanchor="left", font={"size": 10, "color": "gray"})
fig.update_layout(margin={"b": 70})
fig.show()

### 3. 승객 나이 분포\n나이가 기록된 714명의 나이 분포입니다. 자료에서 나이를 확인할 수 없는 승객 177명은 제외합니다.

In [5]:
age_df = df.dropna(subset=["Age"])
fig = px.histogram(age_df, x="Age", nbins=16,
                   title="타이타닉 승객 나이 분포",
                   labels={"Age": "나이 (세)", "count": "승객 수 (명)"},
                   color_discrete_sequence=["#58A5A5"])
fig.update_xaxes(title="나이 (세)")
fig.update_yaxes(title="승객 수 (명)")
fig.add_annotation(text="출처: train.csv · 나이 미상 177명 제외 (n=714)", xref="paper", yref="paper", x=0, y=-0.2, showarrow=False, xanchor="left", font={"size": 10, "color": "gray"})
fig.update_layout(margin={"b": 70})
fig.show()

검수: 실제 기록과 모델 예측을 구분하세요. 전체 탑승자나 인과관계로 일반화하지 않습니다. 개인정보가 있는 이름·표 번호는 출력하지 않습니다. 원출처 이용 조건과 Survived 코드 정의는 제출 전 확인하세요.